# Build Final Public Dataset

Constructs `datasets/final_dataset.csv` — the clean, public-contribution dataset for the paper:

> *A novel structured dataset of CTA price observations parsed from 60 weekly Forbes & Walker broker
> reports (April 2025 – March 2026), enriched with region-specific daily weather from the Open-Meteo
> historical archive and aligned at the sale-week level with lagged weather features for segments.*

**What this notebook does:**
1. Loads `data/processed/tea_preprocessed.csv` (185 columns, 7,100 rows)
2. Drops ML engineering artifacts, internal flags, and redundant columns
3. Adds lagged weather features (lag 1–3) for humidity and wind speed (already present for precipitation, sunshine, and temperature)
4. Renames columns for public readability
5. Writes `datasets/final_dataset.csv`

**Regions covered by Open-Meteo weather:**
- `low_grown` — Low-grown estates (below ~600 m)
- `nuwara_eliya` — Nuwara Eliya high-grown estates
- `uva_udapussellawa` — Uva / Udapussellawa estates
- `western_high` — Western high-grown estates

**Lag convention:** `_lag1` = previous week, `_lag2` = two weeks prior, `_lag3` = three weeks prior

In [2]:
import pandas as pd
import numpy as np

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/tea_preprocessed.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 7,100 rows × 185 columns


## 1. Add Missing Lag Features

Humidity and wind speed exist only for the current week in the source data.
We compute lags at the **sale-week level** (grouped by `sale_number`, sorted chronologically)
then merge back to the row-level dataframe.

In [3]:
# Columns that need lag 1-3 added
NEEDS_LAGS = [
    # Humidity — max and min
    'low_grown__relative_humidity_2m_max_mean',
    'nuwara_eliya__relative_humidity_2m_max_mean',
    'uva_udapussellawa__relative_humidity_2m_max_mean',
    'western_high__relative_humidity_2m_max_mean',
    'low_grown__relative_humidity_2m_min_mean',
    'nuwara_eliya__relative_humidity_2m_min_mean',
    'uva_udapussellawa__relative_humidity_2m_min_mean',
    'western_high__relative_humidity_2m_min_mean',
    # Wind speed
    'low_grown__windspeed_10m_max_mean',
    'nuwara_eliya__windspeed_10m_max_mean',
    'uva_udapussellawa__windspeed_10m_max_mean',
    'western_high__windspeed_10m_max_mean',
]

# Weather values are identical for all rows sharing a sale_number.
# Compute lags on the unique-per-sale-week view, then merge back.
sale_weather = (
    df[['sale_number', 'sale_year', 'sale_month'] + NEEDS_LAGS]
    .drop_duplicates('sale_number')
    .sort_values(['sale_year', 'sale_number'])
    .reset_index(drop=True)
)

new_lag_cols = []
for col in NEEDS_LAGS:
    for lag in [1, 2, 3]:
        lag_col = f'{col}_lag{lag}'
        sale_weather[lag_col] = sale_weather[col].shift(lag)
        new_lag_cols.append(lag_col)

df = df.merge(
    sale_weather[['sale_number'] + new_lag_cols],
    on='sale_number',
    how='left'
)

print(f'Added {len(new_lag_cols)} lag columns → shape now {df.shape}')

Added 36 lag columns → shape now (7100, 221)


## 2. Select Columns

**Dropped:**
- `table_source` — internal pipeline source tag
- `is_production_known`, `has_price_target` — internal boolean flags
- `supply_pressure_index`, `demand_intensity_ratio`, `market_tightness_indicator` — engineered composite indices
- `*_enc` columns — label-encoding artifacts for ML pipelines
- `price_mid_lkr_log` — log transform; can be recomputed
- `rain_sum_total_lag*` — incomplete (no lag-0 baseline); redundant with `precipitation_sum_total`
- `relative_humidity_2m_max_max` — present for only 2 of 4 regions
- `fx_jpy_*` — JPY not a primary pricing currency for Sri Lankan CTA auctions
- `*_todate_*` sales channel columns — year-to-date aggregates; weekly columns are the primary signal

In [4]:
# ── Auction identification ────────────────────────────────────────────────────
IDENTIFIERS = [
    'sale_id', 'sale_number', 'sale_year', 'sale_month', 'sale_date_raw',
]

# ── Tea classification ────────────────────────────────────────────────────────
CLASSIFICATION = [
    'elevation',       # growing elevation: high_grown / medium_grown / low_grown
    'category_type',   # market segment: high_grown / low_grown / off_grade / dust
    'grade',           # orthodox grade (bop, fbop, op, pekoe, …)
    'tier',            # quality tier: select_best / best / below_best / others
    'category',        # full category label
]

# ── Price observations (CTA, LKR per kg) ─────────────────────────────────────
PRICE = [
    'price_lo_lkr',
    'price_hi_lkr',
    'price_mid_lkr',   # primary target variable
    'price_range_lkr',
    'price_mid_usd',
]

# ── Auction volume ────────────────────────────────────────────────────────────
VOLUME = [
    'total_lots',
    'total_kgs',
    'reprint_lots',
    'reprint_quantity',
    'volume_yoy_change_pct',
]

# ── Market sentiment (derived from broker report narrative) ───────────────────
SENTIMENT = [
    'sentiment_overall',
    'sentiment_ex_estate',
    'sentiment_low_grown',
]

# ── Aggregate weather scores (from broker report text) ────────────────────────
BROKER_WEATHER_SCORES = [
    'western_nuwara_eliya_weather_score',
    'uva_udapussellawa_weather_score',
    'low_grown_weather_score',
    'avg_weather_severity',
]

# ── Crop production signals (from broker report) ──────────────────────────────
CROP_PRODUCTION = [
    'crop_nuwara_eliya_trend',
    'crop_western_trend',
    'crop_uva_trend',
    'crop_low_grown_trend',
    'sl_production_mkgs',
]

# ── Category-level demand scores (from broker report) ────────────────────────
DEMAND_SCORES = [
    'dust__demand_score',
    'ex_estate__demand_score',
    'leafy__demand_score',
    'off_grade__demand_score',
    'premium_flowery__demand_score',
    'semi_leafy__demand_score',
]

# ── Category-level quantity sold (million kg) ─────────────────────────────────
QTY_SOLD = [
    'dust__qty_mkgs',
    'ex_estate__qty_mkgs',
    'high_medium__qty_mkgs',
    'leafy__qty_mkgs',
    'off_grade__qty_mkgs',
    'premium_flowery__qty_mkgs',
    'semi_leafy__qty_mkgs',
    'tippy__qty_mkgs',
    'total__qty_mkgs',
]

# ── FX rates (LKR per unit of foreign currency) ───────────────────────────────
FX_RATES = [
    'fx_usd_2025', 'fx_usd_2026',
    'fx_gbp_2025', 'fx_gbp_2026',
    'fx_eur_2025', 'fx_eur_2026',
]

# ── Weekly sales channel volumes (from broker report) ────────────────────────
SALES_CHANNELS = [
    'private_sales_weekly_2025',    'private_sales_weekly_2026',
    'public_auction_weekly_2025',   'public_auction_weekly_2026',
    'forward_contracts_weekly_2025','forward_contracts_weekly_2026',
    'total_sold_weekly_2025',       'total_sold_weekly_2026',
]

# ── Open-Meteo weather features ───────────────────────────────────────────────
# Build programmatically from all weather-related columns in the (now-augmented) df
# Keep: precipitation_sum_total, sunshine_duration_total, temperature_2m_mean_mean,
#        relative_humidity_2m_max_mean, relative_humidity_2m_min_mean,
#        windspeed_10m_max_mean, text_condition_score,
#        text_has_bright, text_has_mist, text_has_rain
#        all_regions__avg_precipitation
# (all variants + lag1/2/3 where available)

WEATHER_KEEP_PATTERNS = [
    'precipitation_sum_total',
    'sunshine_duration_total',
    'temperature_2m_mean_mean',
    'relative_humidity_2m_max_mean',
    'relative_humidity_2m_min_mean',
    'windspeed_10m_max_mean',
    'text_condition_score',
    'text_has_bright',
    'text_has_mist',
    'text_has_rain',
    'all_regions__avg_precipitation',
]

WEATHER_COLS = sorted([
    c for c in df.columns
    if any(pat in c for pat in WEATHER_KEEP_PATTERNS)
])

print(f'Weather columns selected: {len(WEATHER_COLS)}')

ALL_KEEP = (
    IDENTIFIERS + CLASSIFICATION + PRICE + VOLUME +
    SENTIMENT + BROKER_WEATHER_SCORES + CROP_PRODUCTION +
    DEMAND_SCORES + QTY_SOLD + FX_RATES + SALES_CHANNELS +
    WEATHER_COLS
)

# Verify all selected columns exist
missing = [c for c in ALL_KEEP if c not in df.columns]
if missing:
    print('WARNING — columns not found:', missing)
else:
    print(f'All {len(ALL_KEEP)} columns found ✓')

Weather columns selected: 124
All 185 columns found ✓


## 3. Rename for Public Readability

In [5]:
RENAME = {
    # Identifiers
    'sale_date_raw':                       'sale_week_date',

    # Classification
    'category_type':                       'segment',
    'tier':                                'quality_tier',

    # Price
    'price_lo_lkr':                        'price_low_lkr',
    'price_hi_lkr':                        'price_high_lkr',

    # Volume
    'reprint_quantity':                    'reprint_kgs',

    # Broker weather scores — shorter names
    'western_nuwara_eliya_weather_score':  'weather_score_western_nuwara_eliya',
    'uva_udapussellawa_weather_score':     'weather_score_uva_udapussellawa',
    'low_grown_weather_score':             'weather_score_low_grown',
    'avg_weather_severity':                'weather_score_avg_severity',

    # Crop trends
    'crop_nuwara_eliya_trend':             'crop_trend_nuwara_eliya',
    'crop_western_trend':                  'crop_trend_western',
    'crop_uva_trend':                      'crop_trend_uva',
    'crop_low_grown_trend':                'crop_trend_low_grown',
    'sl_production_mkgs':                  'production_sl_mkgs',

    # Demand scores
    'dust__demand_score':                  'demand_score_dust',
    'ex_estate__demand_score':             'demand_score_ex_estate',
    'leafy__demand_score':                 'demand_score_leafy',
    'off_grade__demand_score':             'demand_score_off_grade',
    'premium_flowery__demand_score':       'demand_score_premium_flowery',
    'semi_leafy__demand_score':            'demand_score_semi_leafy',

    # Qty sold
    'dust__qty_mkgs':                      'qty_sold_dust_mkgs',
    'ex_estate__qty_mkgs':                 'qty_sold_ex_estate_mkgs',
    'high_medium__qty_mkgs':               'qty_sold_high_medium_mkgs',
    'leafy__qty_mkgs':                     'qty_sold_leafy_mkgs',
    'off_grade__qty_mkgs':                 'qty_sold_off_grade_mkgs',
    'premium_flowery__qty_mkgs':           'qty_sold_premium_flowery_mkgs',
    'semi_leafy__qty_mkgs':                'qty_sold_semi_leafy_mkgs',
    'tippy__qty_mkgs':                     'qty_sold_tippy_mkgs',
    'total__qty_mkgs':                     'qty_sold_total_mkgs',

    # FX rates
    'fx_usd_2025': 'fx_usd_lkr_2025',  'fx_usd_2026': 'fx_usd_lkr_2026',
    'fx_gbp_2025': 'fx_gbp_lkr_2025',  'fx_gbp_2026': 'fx_gbp_lkr_2026',
    'fx_eur_2025': 'fx_eur_lkr_2025',  'fx_eur_2026': 'fx_eur_lkr_2026',

    # Sales channels
    'private_sales_weekly_2025':       'sales_private_weekly_2025',
    'private_sales_weekly_2026':       'sales_private_weekly_2026',
    'public_auction_weekly_2025':      'sales_public_auction_weekly_2025',
    'public_auction_weekly_2026':      'sales_public_auction_weekly_2026',
    'forward_contracts_weekly_2025':   'sales_forward_contracts_weekly_2025',
    'forward_contracts_weekly_2026':   'sales_forward_contracts_weekly_2026',
    'total_sold_weekly_2025':          'sales_total_weekly_2025',
    'total_sold_weekly_2026':          'sales_total_weekly_2026',
}

final_df = df[ALL_KEEP].rename(columns=RENAME)
print(f'Final dataset: {final_df.shape[0]:,} rows × {final_df.shape[1]} columns')

Final dataset: 7,100 rows × 185 columns


## 4. Data Quality Check

In [6]:
null_pct = (final_df.isna().sum() / len(final_df) * 100).round(1)
has_nulls = null_pct[null_pct > 0].sort_values(ascending=False)

print('=== Columns with missing values ===')
if len(has_nulls) == 0:
    print('None — dataset is complete.')
else:
    print(has_nulls.to_string())

print(f'\n=== Segment distribution ===')
print(final_df['segment'].value_counts())

print(f'\n=== Sale week range ===')
print(final_df[['sale_year', 'sale_number', 'sale_week_date']].drop_duplicates()
      .sort_values(['sale_year', 'sale_number'])
      .head(3)
      .to_string(index=False))
print('...')
print(final_df[['sale_year', 'sale_number', 'sale_week_date']].drop_duplicates()
      .sort_values(['sale_year', 'sale_number'])
      .tail(3)
      .to_string(index=False))

print(f'\n=== Price summary (LKR/kg) ===')
print(final_df['price_mid_lkr'].describe().round(2))

=== Columns with missing values ===
quality_tier                                             57.6
grade                                                    37.0
nuwara_eliya__relative_humidity_2m_max_mean_lag3         29.8
low_grown__relative_humidity_2m_max_mean_lag3            29.8
western_high__relative_humidity_2m_min_mean_lag3         29.8
western_high__relative_humidity_2m_max_mean_lag3         29.8
uva_udapussellawa__windspeed_10m_max_mean_lag3           29.8
uva_udapussellawa__relative_humidity_2m_min_mean_lag3    29.8
uva_udapussellawa__relative_humidity_2m_max_mean_lag3    29.8
nuwara_eliya__windspeed_10m_max_mean_lag3                29.8
nuwara_eliya__relative_humidity_2m_min_mean_lag3         29.8
low_grown__windspeed_10m_max_mean_lag3                   29.8
low_grown__relative_humidity_2m_min_mean_lag3            29.8
western_high__windspeed_10m_max_mean_lag3                29.8
western_high__relative_humidity_2m_max_mean_lag2         26.5
uva_udapussellawa__relative_humidi

## 5. Column Inventory

In [7]:
groups = {
    'Auction identification':       IDENTIFIERS,
    'Tea classification':           CLASSIFICATION,
    'Price (LKR/kg, USD/kg)':       PRICE,
    'Auction volume':               VOLUME,
    'Market sentiment':             SENTIMENT,
    'Broker weather scores':        BROKER_WEATHER_SCORES,
    'Crop & production signals':    CROP_PRODUCTION,
    'Category demand scores':       DEMAND_SCORES,
    'Category quantity sold (mkgs)':QTY_SOLD,
    'FX rates (LKR per unit)':      FX_RATES,
    'Weekly sales channels':        SALES_CHANNELS,
    'Open-Meteo weather features':  WEATHER_COLS,
}

# Use post-rename names where applicable
for grp, cols in groups.items():
    renamed = [RENAME.get(c, c) for c in cols]
    # Weather cols are not renamed
    print(f'\n{grp} ({len(cols)})')
    for c in renamed:
        print(f'  {c}')


Auction identification (5)
  sale_id
  sale_number
  sale_year
  sale_month
  sale_week_date

Tea classification (5)
  elevation
  segment
  grade
  quality_tier
  category

Price (LKR/kg, USD/kg) (5)
  price_low_lkr
  price_high_lkr
  price_mid_lkr
  price_range_lkr
  price_mid_usd

Auction volume (5)
  total_lots
  total_kgs
  reprint_lots
  reprint_kgs
  volume_yoy_change_pct

Market sentiment (3)
  sentiment_overall
  sentiment_ex_estate
  sentiment_low_grown

Broker weather scores (4)
  weather_score_western_nuwara_eliya
  weather_score_uva_udapussellawa
  weather_score_low_grown
  weather_score_avg_severity

Crop & production signals (5)
  crop_trend_nuwara_eliya
  crop_trend_western
  crop_trend_uva
  crop_trend_low_grown
  production_sl_mkgs

Category demand scores (6)
  demand_score_dust
  demand_score_ex_estate
  demand_score_leafy
  demand_score_off_grade
  demand_score_premium_flowery
  demand_score_semi_leafy

Category quantity sold (mkgs) (9)
  qty_sold_dust_mkgs
  qty_s

## 6. Save

In [8]:
out_path = '../datasets/final_dataset.csv'
final_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape : {final_df.shape[0]:,} rows × {final_df.shape[1]} columns')

Saved → ../datasets/final_dataset.csv
Shape : 7,100 rows × 185 columns
